# Position Sizing in Python: Kelly Criterion + Fixed-Fractional

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mindgaptech/algodrill-notebooks/blob/main/notebooks/position_sizing.ipynb)

Companion notebook to [algodrill.app/code/position-sizing-python](https://algodrill.app/code/position-sizing-python). ~25 lines of numpy/pandas: full Kelly, half-Kelly, quarter-Kelly, and fixed-fractional risk from a worked example (55% win rate, $150 avg win, $100 avg loss), plus a batch Kelly computation from a raw trade P&L series.

**Nothing here is investment advice.** This is a position-sizing formula, not a recommendation for how much you personally should risk. See [algodrill.app/kelly-criterion](https://algodrill.app/kelly-criterion) for the full derivation and failure modes before using Kelly sizing on a real account.

## Setup

Run this cell first if you're in Colab (installs the pinned versions below into the Colab runtime). If you're running locally with the versions already installed, skip it.

In [1]:
# Colab only -- uncomment to install. Local runs should already have these
# pinned via: uv venv .venv && uv pip install -p .venv numpy==2.4.6 pandas==2.3.3
# !pip install -q numpy==2.4.6 pandas==2.3.3

## Kelly sizing, fixed-fractional, and batch Kelly from a P&L series

Pure math -- no market data download required. The worked example (55% WR, $150 avg win, $100 avg loss) matches the numbers used on the Kelly Criterion guide, so the two pages agree.

In [2]:
"""AlgoDrill — Kelly fraction and fixed-fractional position sizing.

~25 lines of numpy/pandas to compute full Kelly, half-Kelly,
quarter-Kelly, and fixed-fractional risk from a trade series.
Pure math — no broker, no execution.

Versions: numpy 2.4.6 · pandas 2.3.3 (run 2026-06-05)
Install:  pip install numpy pandas
"""
import numpy as np
import pandas as pd


def kelly_fraction(win_rate: float, avg_win: float, avg_loss: float) -> dict:
    """Kelly fraction from win rate and average payoffs."""
    loss_rate  = 1.0 - win_rate
    expectancy = win_rate * avg_win - loss_rate * avg_loss  # E[trade]
    kelly_f    = expectancy / avg_win                       # f* = E/W
    return {
        "expectancy":   round(expectancy, 4),
        "payoff_ratio": round(avg_win / avg_loss, 4),
        "kelly_f":      round(kelly_f, 4),
        "half_kelly":   round(kelly_f / 2, 4),
        "qtr_kelly":    round(kelly_f / 4, 4),
    }


# ── Example: 55% WR, $150 avg win, $100 avg loss (matches /kelly-criterion) ──
r = kelly_fraction(0.55, 150.0, 100.0)
print("=== Kelly sizing: 55% WR, $150 avg win, $100 avg loss ===")
print(f"Expectancy      ${r['expectancy']:.2f} per trade")
print(f"Payoff ratio      {r['payoff_ratio']:.4f}")
print(f"Full Kelly        {r['kelly_f']:.1%}")
print(f"Half-Kelly        {r['half_kelly']:.1%}")
print(f"Quarter-Kelly     {r['qtr_kelly']:.1%}")
print()

# ── Fixed-fractional comparison on a $10,000 account ─────────────────────────
ACCOUNT = 10_000
print(f"=== Fixed-fractional risk on ${ACCOUNT:,} account ===")
for pct in (0.01, 0.02, r["half_kelly"], r["kelly_f"]):
    risk = ACCOUNT * pct
    print(f"  {pct:.1%}  → ${risk:>7.2f} at risk per trade")
print()

# ── Batch Kelly from a real trade P&L series (pandas path) ───────────────────
pnl = pd.Series([150, -100, 200, -90, 175, -110, 300, -95, 225, -100])
wins   = pnl[pnl > 0]
losses = pnl[pnl < 0]
wr     = len(wins) / len(pnl)
r2 = kelly_fraction(wr, float(wins.mean()), float(abs(losses.mean())))
print(f"=== Batch Kelly: 10-trade P&L series ===")
print(f"Trades {len(pnl)}  WR {wr:.0%}  Avg win ${wins.mean():.2f}  Avg loss ${abs(losses.mean()):.2f}")
print(f"Full Kelly        {r2['kelly_f']:.1%}")
print(f"Half-Kelly        {r2['half_kelly']:.1%}")


=== Kelly sizing: 55% WR, $150 avg win, $100 avg loss ===
Expectancy      $37.50 per trade
Payoff ratio      1.5000
Full Kelly        25.0%
Half-Kelly        12.5%
Quarter-Kelly     6.2%

=== Fixed-fractional risk on $10,000 account ===
  1.0%  → $ 100.00 at risk per trade
  2.0%  → $ 200.00 at risk per trade
  12.5%  → $1250.00 at risk per trade
  25.0%  → $2500.00 at risk per trade

=== Batch Kelly: 10-trade P&L series ===
Trades 10  WR 50%  Avg win $210.00  Avg loss $99.00
Full Kelly        26.4%
Half-Kelly        13.2%


## Reading the sizing output

Full Kelly (25.0%) maximizes long-run geometric growth but implies 30-50% drawdowns during bad streaks. Half-Kelly (12.5%) captures roughly 75% of full-Kelly growth at about half the drawdown -- the standard practitioner starting point. The batch section shows the same formula applied to a raw 10-trade P&L series, which is closer to how you would actually compute this from your own trade log.

Two pages go deeper:

- [Position Sizing walkthrough](https://algodrill.app/code/position-sizing-python) -- the full line-by-line explanation and a when-to-use-each-approach table.
- [Kelly Criterion guide](https://algodrill.app/kelly-criterion) -- the derivation, the relationship to Sharpe Ratio, and failure modes like multi-position correlation.

---
Back to [All Code Walkthroughs](https://algodrill.app/code) &middot; [algodrill-notebooks on GitHub](https://github.com/mindgaptech/algodrill-notebooks)